In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

# 1. Load Data
try:
    df = pd.read_csv('data/data_platform-strategy-list_1769496701.csv')
except:
    # Fallback if file isn't found in this session (though it should be)
    print("Error loading file.")

# 2. Preprocessing & Meta Extraction
def extract_mode(strategy_id):
    match = re.search(r'mode_(\d+)$', str(strategy_id))
    return int(match.group(1)) if match else 0

df['mode_index'] = df['调度名'].apply(extract_mode)
df['symbol'] = df['币对(标准)']

# Numeric conversion
cols_numeric = ['总收益', '成交收益', '资金费收益', '手续费', '滑点', 
                '当前持仓', '最大持仓', '敞口', '市场成交占比', '成交量']
for c in cols_numeric:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

# Date conversion
df['调度最后更新时间'] = pd.to_datetime(df['调度最后更新时间'], errors='coerce')
df['调度最后成交时间'] = pd.to_datetime(df['调度最后成交时间'], errors='coerce')

# 3. Aggregation by Symbol
# Sort by symbol and mode to easily get the 'latest' state
df = df.sort_values(by=['symbol', 'mode_index'])

def agg_func(x):
    latest = x.iloc[-1]
    
    # Summations for cumulative metrics
    total_spread = x['成交收益'].sum()
    total_commission = x['手续费'].sum()
    total_slippage = x['滑点'].sum() # usually negative for loss
    
    # Funding: User logic says "Fund Income field directly reads the cumulative value".
    # So we take the latest value. 
    # Logic check: If mode_N takes over mode_N-1, does the account funding reset?
    # User said: "Different modes might show same value (account total), cannot sum across modes."
    # Implication: The latest mode's funding column likely holds the total funding for that coin on that account.
    total_funding = latest['资金费收益']
    
    # Recalculate Total PnL to be consistent
    # Total PnL = Spread (Sum) + Funding (Latest) + Slippage (Sum) - Commission (Sum)
    # Note: 'Slippage' column is often negative in data. Let's assume it is signed.
    # Commission is usually positive cost.
    # Standard PnL = Spread + Funding + Slippage_Signed - Commission
    total_pnl = total_spread + total_funding + total_slippage - total_commission
    
    return pd.Series({
        'Mode次数': x['mode_index'].max(),
        '总收益': total_pnl,
        '成交收益(价差)': total_spread,
        '资金费收益': total_funding,
        '手续费': total_commission,
        '滑点': total_slippage,
        '当前持仓': latest['当前持仓'],
        '最大持仓': x['最大持仓'].max(),
        '敞口': latest['敞口'],
        '最后更新时间': latest['调度最后更新时间'],
        '最后成交时间': latest['调度最后成交时间'],
        '市场成交占比': latest['市场成交占比']
    })

df_res = df.groupby('symbol').apply(agg_func).reset_index()

# 4. Derived Metrics

# 4.1 Cost Efficiency (Cost Ratio)
# Ratio = (Commission + |Slippage|) / Total PnL
# Only relevant for profitable strategies to see how much "gross profit" is eaten.
df_res['成本'] = df_res['手续费'] + df_res['滑点'].abs()
df_res['成本损耗率'] = np.where(df_res['总收益'] > 0, df_res['成本'] / df_res['总收益'], np.nan)

# 4.2 PnL Source
def classify_source(row):
    if row['总收益'] <= 0:
        return '亏损'
    # If Spread is negative but Total is positive -> Funding subsidized it
    if row['成交收益(价差)'] < 0:
        return '资金费补偿(价差亏损)'
    # If both positive, check which is larger
    if row['资金费收益'] > row['成交收益(价差)']:
        return '资金费驱动'
    else:
        return '价差驱动'

df_res['盈利模式'] = df_res.apply(classify_source, axis=1)

# 4.3 Lifecycle Status
# Using Max Pos as baseline. 
# > 80% -> Full/Building
# < 10% -> Closing/Closed
df_res['持仓比例'] = (df_res['当前持仓'].abs() / df_res['最大持仓'].abs()).fillna(0)
def classify_status(row):
    if abs(row['当前持仓']) < 1e-4: # effectively 0
        return '已平仓/结束'
    if row['持仓比例'] < 0.1:
        return '平仓尾声'
    return '持仓/建仓中'

df_res['状态'] = df_res.apply(classify_status, axis=1)

# 4.4 Stale Check
# Assuming "Now" is the max date in the dataset to avoid timezone issues with real 'now'
current_data_time = df_res['最后更新时间'].max()
stale_threshold = timedelta(hours=1)
df_res['是否异常停滞'] = (current_data_time - df_res['最后更新时间']) > stale_threshold

# 5. Output Tables Generation

# Table 1: Top Winners
df_winners = df_res.sort_values(by='总收益', ascending=False).head(10)
cols_win = ['symbol', '总收益', '盈利模式', '成本损耗率', '状态', 'Mode次数']

# Table 2: Top Losers
df_losers = df_res.sort_values(by='总收益', ascending=True).head(10)
cols_loss = ['symbol', '总收益', '成交收益(价差)', '资金费收益', '滑点', 'Mode次数']

# Table 3: High Cost Ratio (The "Hard Earned" Money)
# Filter: Profitable AND Cost Ratio > 0.3 (30% of profit goes to fees/slip)
df_costly = df_res[(df_res['总收益'] > 100) & (df_res['成本损耗率'] > 0.3)].sort_values(by='成本损耗率', ascending=False).head(5)
cols_cost = ['symbol', '总收益', '成本', '成本损耗率', '盈利模式']

# Table 4: High Frequency Modes (Instability)
df_freq = df_res[df_res['Mode次数'] >= 5].sort_values(by='Mode次数', ascending=False).head(10)
cols_freq = ['symbol', 'Mode次数', '总收益', '状态']

# Table 5: Stale Strategies
df_stale = df_res[df_res['是否异常停滞'] == True].sort_values(by='最后更新时间')
cols_stale = ['symbol', '最后更新时间', '状态']

# Table 6: Closing Phase (Watchlist for completion)
df_closing = df_res[df_res['状态'] == '平仓尾声'].sort_values(by='持仓比例')
cols_closing = ['symbol', '持仓比例', '当前持仓', '总收益']

# Print outputs for capture
print("=== WINNERS ===")
print(df_winners[cols_win].to_string(index=False))
print("\n=== LOSERS ===")
print(df_losers[cols_loss].to_string(index=False))
print("\n=== HIGH COST EFFICIENCY ===")
print(df_costly[cols_cost].to_string(index=False))
print("\n=== HIGH FREQ MODES ===")
print(df_freq[cols_freq].to_string(index=False))
print("\n=== STALE STRATEGIES ===")
print(df_stale[cols_stale].head(10).to_string(index=False)) # Show top 10 stale
print("\n=== CLOSING PHASE ===")
print(df_closing[cols_closing].to_string(index=False))
print("\n=== RISK EXPOSURE ===")
# Show if any exposure is large (e.g. > 1000 USDT value? Data is qty. Assuming price is approx 1 for stable, but these are altcoins. 
# Just show top absolute exposure values)
print(df_res[['symbol', '敞口']].sort_values(by='敞口', key=abs, ascending=False).head(5).to_string(index=False))

=== WINNERS ===
           symbol         总收益        盈利模式    成本损耗率     状态  Mode次数
     XMR-USDT-FTR 7450.832996       资金费驱动 0.021501 持仓/建仓中       2
       H-USDT-FTR 2738.112350 资金费补偿(价差亏损) 0.070834 持仓/建仓中       1
    HYPE-USDT-FTR 2232.334502       资金费驱动 0.099607 持仓/建仓中       3
    KITE-USDT-FTR 1917.700518       资金费驱动 0.000232 持仓/建仓中       1
     LIT-USDT-FTR 1596.884380       资金费驱动 0.048742 持仓/建仓中       0
     CYS-USDT-FTR 1249.358406       资金费驱动 0.002246 持仓/建仓中       0
FARTCOIN-USDT-FTR 1147.657371 资金费补偿(价差亏损) 0.202806 持仓/建仓中       2
   LIGHT-USDT-FTR 1059.808710       资金费驱动 0.151341 持仓/建仓中       0
    NEAR-USDT-FTR  969.857193        价差驱动 0.118497 持仓/建仓中       4
   ASTER-USDT-FTR  928.279203        价差驱动 0.350137 持仓/建仓中       1

=== LOSERS ===
         symbol          总收益     成交收益(价差)      资金费收益           滑点  Mode次数
STABLE-USDT-FTR -2689.729936 -2101.598360 328.605925  -870.178632       2
   BNB-USDT-FTR -2279.211445 -2045.798507 421.874968  -357.043922       0
  MERL-USDT-FTR -195